In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
!pip install -q huggingface_hub

import os
from google.colab import userdata
from huggingface_hub import login

# Lay token tu Secrets cua Colab
hf_token = userdata.get('tuankun08')

# Tien hanh dang nhap
login(token=hf_token)
print("Da ket noi thanh cong voi Hugging Face!")

In [ ]:
from huggingface_hub import snapshot_download

repo_id = "tuankun08/Dataset"

# Tai toan bo repository/folder ve thu muc '/content/Dataset' tren Colab
snapshot_download(
    repo_id=repo_id,
    allow_patterns="Dataset/*",
    repo_type="dataset",
    local_dir="/content/Dataset"
)

In [ ]:
!git clone https://github.com/tuananhh08/Hallsensor_final.git /content/Hallsensor_final
!pip install -q optuna scikit-learn pandas numpy matplotlib

In [ ]:
DATA_DIR  = "/content/Dataset/Dataset"
REPO      = "/content/Hallsensor_final/Benchmark/CRLocNet"
RUNS      = "/content/drive/MyDrive/CRLocNet_runs"
CACHE     = "/content/crlocnet_cache"

# Finetune data
RAW_VOLTAGE           = f"{DATA_DIR}/Grid_add_Random_data.csv"
CLEAN_VOLTAGE         = f"{DATA_DIR}/Grid_add_Random_data_computed.csv"
RAW_LABEL             = f"{DATA_DIR}/Grid_add_Random_points_coordinates.csv"

# Grid data
GRID_VOLTAGE          = f"{DATA_DIR}/Grid_data.csv"
GRID_CLEAN_VOLTAGE    = f"{DATA_DIR}/Grid_data_computed.csv"
GRID_LABEL            = f"{DATA_DIR}/Grid_points_coordinates.csv"

# Helix data (dung cho evaluation)
HELIX_VOLTAGE         = f"{DATA_DIR}/Helix_data_2.csv"
HELIX_CLEAN_VOLTAGE   = f"{DATA_DIR}/Helix_data_2_computed.csv"
HELIX_LABEL           = f"{DATA_DIR}/Helix_points_coordinates_2.csv"

# Random data
RANDOM_VOLTAGE        = f"{DATA_DIR}/Random_data.csv"
RANDOM_CLEAN_VOLTAGE  = f"{DATA_DIR}/Random_data_computed.csv"
RANDOM_LABEL          = f"{DATA_DIR}/Random_points_coordinates.csv"

# Synthetic data ~700k (Stage 2)
SYNTHETIC_VOLTAGE = f"{DATA_DIR}/synthetic_grid_data.csv"
SYNTHETIC_LABEL   = f"{DATA_DIR}/synthetic_grid_coordinates.csv"

# Calibration parameters path
CALIB_PHYSICAL_CSV = f"{DATA_DIR}/Calibration_Parameters_r_per_sensor.csv"
CALIB_ALPHA_CSV    = f"{DATA_DIR}/Calibration_Alpha_r_per_sensor.csv"

# Theoretical EPM CSVs (tuy chon - de None neu khong co)
# Moi file phai co cung so hang voi data tuong ung, 64 cot (flat EPM field).
RAW_EPM       = None   # e.g. f"{DATA_DIR}/real_epm_64col.csv"
SYNTHETIC_EPM = None   # e.g. f"{DATA_DIR}/synth_epm_64col.csv"
HELIX_EPM     = None   # e.g. f"{DATA_DIR}/helix_epm_64col.csv"

print("Duong dan da khai bao:")
print(f"  DATA_DIR  = {DATA_DIR}")
print(f"  REPO      = {REPO}")
print(f"  RUNS      = {RUNS}")
print(f"  CACHE     = {CACHE}")

In [ ]:
# STAGE 1 - Pre-train ModNet
# ModNet hoc noise suppression + domain alignment tu synthetic data co them Gaussian noise.
!python $REPO/run.py \
  --stage modnet \
  --raw-voltage     "$RAW_VOLTAGE" \
  --clean-voltage   "$CLEAN_VOLTAGE" \
  --raw-label       "$RAW_LABEL" \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label   "$SYNTHETIC_LABEL" \
  --ckpt-dir   "$RUNS/modnet" \
  --cache-dir  "$CACHE" \
  --batch-size 128 \
  --epochs     200 \
  --lr         1e-4 \
  --weight-decay 4.56e-3 \
  --noise-std  0.1 \
  --warmup-epochs 5 \
  --eta-min    1e-6 \
  --patience   45 \
  --save-every 10 \
  --grad-clip  1.0 \
  --no-compile

In [ ]:
# STAGE 2 - Pre-train LocNet tren synthetic data.
# Raw/clean paths duoc truyen vao de dung chung cache/scaler; LocNet chi hoc synthetic data.
!python $REPO/run.py \
  --stage locnet \
  --raw-voltage "$RAW_VOLTAGE" \
  --clean-voltage "$CLEAN_VOLTAGE" \
  --raw-label "$RAW_LABEL" \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label "$SYNTHETIC_LABEL" \
  --ckpt-dir "$RUNS/locnet" \
  --cache-dir "$CACHE" \
  --batch-size 512 \
  --epochs 200 \
  --lr 1e-4 \
  --weight-decay 4.56e-3 \
  --lambda-direction 0.2 \
  --warmup-epochs 5 \
  --eta-min 1e-6 \
  --patience 45 \
  --save-every 10 \
  --grad-clip 1.0 \
  --no-compile

In [ ]:
# STAGE 3 - Fine-tune CRLocNet (ModNet + LocNet) tren real data
!python $REPO/run.py \
  --stage finetune \
  --raw-voltage "$RAW_VOLTAGE" \
  --clean-voltage "$CLEAN_VOLTAGE" \
  --raw-label "$RAW_LABEL" \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label "$SYNTHETIC_LABEL" \
  --modnet-checkpoint "$RUNS/modnet/modnet_pretrained.pt" \
  --locnet-checkpoint "$RUNS/locnet/locnet_pretrained.pt" \
  --ckpt-dir "$RUNS/finetune" \
  --cache-dir "$CACHE" \
  --batch-real 32 \
  --epochs 200 \
  --lr-modnet 1e-4 \
  --lr-locnet 1e-4 \
  --weight-decay 4.56e-3 \
  --lambda-direction 0.2 \
  --lambda-physics 1e-4 \
  --lambda-alignment 0.1 \
  --warmup-epochs 5 \
  --eta-min 1e-6 \
  --patience 45 \
  --save-every 10 \
  --grad-clip 1.0 \
  --no-compile

In [ ]:
import os
EVAL_DIR = f"{RUNS}/evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)

!python $REPO/evaluation/evaluate.py \
  --test-voltage "$HELIX_VOLTAGE" \
  --test-label "$HELIX_LABEL" \
  --ckpt-dir "$RUNS/finetune" \
  --checkpoint "full_model_best.pt" \
  --scalers "$RUNS/finetune/scalers.pkl" \
  --batch-size 64 \
  --device cuda \
  --out "$RUNS/evaluation/crlocnet_trajectory.png" \
  --out-csv "$RUNS/evaluation/crlocnet_predictions.csv"

from IPython.display import display, Image
display(Image(filename=f"{EVAL_DIR}/crlocnet_trajectory.png"))
display(Image(filename=f"{EVAL_DIR}/crlocnet_position_error.png"))
display(Image(filename=f"{EVAL_DIR}/crlocnet_orientation_error.png"))